# Диффузионные модели (Diffusion Models)

**темы:** развитие идеи латентных моделей, forward и reverse процессы, DDPM, параметризация и noise-prediction loss, U-Net, диффузия на непрерывном времени (SDE, уравнение Фоккера–Планка), DDIM, Stable Diffusion, CLIP, DALL·E

**Автор:** Федоров Артем Максимович

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

# Позиционирование диффузионных моделей

На предыдущих лекциях мы познакомились с двумя ключевыми классическими типами архитектур:

- **GAN** — неявная генеративная модель, обучение которой осуществляется через состязетельный процесс "игры" генератора и дискриминатора. Даёт визуально чёткие сэмплы, но страдает от mode collapse и не позволяет оценить плотность настоящего распределения через $p_\theta(x) \approx p^\ast(x) $.

- **VAE** — простейшая латентная модель, вводящая латентное пространство $\mathcal{Z}$ и обучаемая через некоторый функционал, приближающий оригинальную задачу максимизации вероятности пронаблюдать выборку $\log p^\ast(x) \approx \log p_\theta(x) \to \max$ (имеется в виду вариационная нижняя оценка ELBO). Даёт нормированную плотность $p_\theta(x)$, но генерирует размытые объекты из-за ограниченной возможности как энкодера, так и декодера запомнить правила отображения сложных объектов в "простое" пространство. 

Оба подхода занимают свои ниши в фундаментальном трейдоффе генеративного моделирования: **качество сэмплов**, **скорость генерации** и **покрытие мод** распределения $p^*$.

<p align="center">
  <img src="images/lecture_4/gan_diff_vae.png" style="width:70%;">
</p>

## Развитие латентных моделей

Естественный вопрос: возможно ли совместить **высокое качество** генерации (как у GAN) с **хорошим покрытием мод** (как у VAE). Отдельно, конечно, хотелось бы еще и распределение оригинальных данных оценивать, так как это много где может быть использовано? 

Ответ — **да**, но за это придётся заплатить **скоростью инференса**. Именно такой компромисс реализуют **диффузионные модели** (Diffusion Models), которые за последние годы стали стандартом де-факто в генерации изображений, аудио, видео и 3D-контента.

> Ключевая идея, которую мы будем развивать на протяжении лекции: **диффузионная модель — это обобщение VAE**, в котором вместо одного «сложного» шага декодирования $z \to x$ используется **длинная цепочка простых шагов** $z_T \to z_{T-1} \to \cdots \to z_0 \approx x$. Каждый отдельный шаг — элементарный и хорошо обусловленный, а сложность моделируемого распределения возникает из композиции.

# Часть 1. Диффузия как обобщение VAE (дискретное время)

При выводе VAE мы вводили латентную переменную $z \in \mathcal{Z}$ и строили генеративную модель в виде:

$$p_\theta(x, z) = p_\theta(x \mid z) \, p(z), \quad p(z) = \mathcal{N}(0, I)$$

Обучение сводилось к максимизации ELBO:

$$\log p^\ast(x) \approx \log p_\theta(x) \geq\mathcal{L}_{\text{ELBO}}(x; \theta, \phi) = \mathbb{E}_{q_\phi(z|x)}\left[\log p_\theta(x \mid z)\right] - \text{KL}\left(q_\phi(z \mid x) \,\|\, p(z)\right)$$

$$\mathcal{L}_{\text{ELBO}}(x; \theta, \phi) \to \max \Longrightarrow \log p_\theta(x) \to \max$$

Такая модель обладает простой интерпретацией: **энкодер** $q_\phi(z \mid x)$ кодирует объект в латент, **декодер** $p_\theta(x \mid z)$ восстанавливает объект из латента

- $\mathbb{E}_{q_\phi(z|x)}\left[\log p_\theta(x \mid z)\right]$ – отвечает за качество реконструкции – насколько хорошо модель способна выучить латентное пространство / отображать в него реальные объекты.
- $\text{KL}\left(q_\phi(z \mid x) \,\|\, p(z)\right)$ – отвечает за интерпретабильно латентного пространства – насколько распределение точек в латентном пространстве сильно отличается от того, каким мы его моделируем, тем самым позволяя оценивать плотность $p^\ast$ и семплировать из распределения.



## В чём проблема VAE

Вся сложность распределения данных $p^*$ должна быть «захвачена» единственным шагом декодера $p_\theta(x \mid z)$. При простом прайоре $p(z) = \mathcal{N}(0, I)$ это означает, что нейросетевой декодер должен научиться превращать Гауссовский шум в сложное многомодальное распределение реальных объектов — **за один шаг**.

На практике это приводит к двум сценариям:
1. Декодер недостаточно мощный — как мы уже разбирали на предыдущей лекции, генерируемые объекты становятся размытыми, усреднениями некоторых "множеств" объектов обучающей выборки, моды не покрываются и вообще все плохо.
2. Декодер избыточно мощный — перепараметризация ведет к плохой генерализации (обобщающей способности) $\Rightarrow$ не получится хорошо работать с новыми/генерируемыми данными, оптимизация ELBO становится плохо обусловленной, обучение нестабильно. 

> Да, стоит отметить, что сейчас начали появляться новые подходы, что выучивают сложное отображение между двумя распределениями за один шаг, при чем достаточно хорошо, но они все еще не способны состязаться с диффузионками, о которых далее пойдет речь. К примеру такие статьи:
> 
> 1) [Generative Modeling via Drifting](https://arxiv.org/abs/2602.04770)
> 2) [One Step Diffusion via Shortcut Models](https://arxiv.org/abs/2410.12557)
> 3) [On the Design of One-step Diffusion via Shortcutting Flow Paths](https://arxiv.org/abs/2512.11831)
> 4) [SANA-Sprint: One-Step Diffusion with Continuous-Time Consistency Distillation](https://arxiv.org/abs/2503.09641)



Интуитивно, мы пытаемся вобрать всю сложность данных одним простое распределение $p(z)$ через слишком сложны единственный шаг декодера $p_\theta(x \mid z)$, что сделать нельзя либо из-за простоты декодера, либо из-за излишней сложности последнего мы просто не можем это выучить.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)

# Демонстрация: сложное мультимодальное распределение vs. один гауссиан
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Истинное распределение -- смесь гауссиан
means = [-3, 0, 4]
stds  = [0.5, 0.3, 0.7]
weights = [0.3, 0.4, 0.3]

x_grid = np.linspace(-6, 8, 500)
p_true = sum(w * norm.pdf(x_grid, m, s) for w, m, s in zip(weights, means, stds))

axes[0].fill_between(x_grid, p_true, alpha=0.3, color='C0')
axes[0].plot(x_grid, p_true, 'C0', lw=2)
axes[0].set_title(r'$p^*(x)$ (3 моды)', fontsize=13)
axes[0].set_xlabel('x')
axes[0].set_ylabel('density')

# 2) Лучшее приближение одним гауссианом
mu_fit = sum(w * m for w, m in zip(weights, means))
var_fit = sum(w * (s**2 + m**2) for w, m, s in zip(weights, means, stds)) - mu_fit**2
p_gauss = norm.pdf(x_grid, mu_fit, np.sqrt(var_fit))

axes[1].fill_between(x_grid, p_gauss, alpha=0.3, color='C1')
axes[1].plot(x_grid, p_gauss, 'C1', lw=2)
axes[1].plot(x_grid, p_true, 'C0--', lw=1.5, alpha=0.5)
axes[1].set_title('VAE: один гауссов декодер', fontsize=13)
axes[1].set_xlabel('x')

# 3) Приближение цепочкой простых шагов (идея диффузии)
T = 5
alphas = np.linspace(0.05, 1.0, T)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, T))
for i, alpha in enumerate(alphas):
    p_interp = alpha * p_true + (1 - alpha) * norm.pdf(x_grid, 0, 2.5)
    p_interp /= np.trapz(p_interp, x_grid)
    axes[2].plot(x_grid, p_interp, color=colors[i], lw=1.5,
                 label=f'step {i+1}/{T}', alpha=0.7 + 0.3*alpha)

axes[2].set_title('Diffusion: gradual transition', fontsize=13)
axes[2].set_xlabel('x')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

На правом графике видна ключевая идея: вместо одного «скачка» от простого распределения к сложному, мы выстраиваем **последовательность промежуточных распределений**, каждое из которых лишь немного отличается от предыдущего. Каждый отдельный переход — простой, а сложность возникает из их композиции.

## 1.2 Цепочка латентных переменных: от VAE к диффузии

### Ключевая идея

Вместо одной латентной переменной $z$ введём **цепочку** латентных переменных:

$$z_T \to z_{T-1} \to \cdots \to z_1 \to z_0 \approx x$$

где $z_T$ — чистый шум, а $z_0$ — объект, близкий к данным. Каждый переход $z_t \to z_{t-1}$ реализуется простым параметрическим распределением (как правило, гауссовым).

Формально мы по-прежнему находимся в рамках латентного подхода, только теперь латентное пространство $\mathcal{Z}$ расширяется до $\mathcal{Z}^T = \mathcal{Z}_1 \times \cdots \times \mathcal{Z}_T$, и совместная генеративная модель записывается как:

$$p_\theta(x, z_{1:T}) = p(z_T) \prod_{t=1}^{T} p_\theta(z_{t-1} \mid z_t), \quad z_0 := x$$

### Forward (прямой) процесс — добавление шума

В отличие от VAE, где приближённый апостериор $q_\phi(z \mid x)$ параметризуется обучаемой нейросетью, в диффузионной модели «энкодер» задаётся **явно и без обучаемых параметров**. Это фиксированный марковский процесс постепенного зашумления:

$$q(z_{1:T} \mid x) = \prod_{t=1}^{T} q(z_t \mid z_{t-1}), \quad q(z_t \mid z_{t-1}) = \mathcal{N}\!\left(z_t;\, \sqrt{\alpha_t}\, z_{t-1},\, (1 - \alpha_t)\, I\right)$$

где $\alpha_t \in (0, 1)$ — заранее выбранные коэффициенты, определяемые **расписанием шума** (noise schedule). На каждом шаге мы:
1. Немного масштабируем сигнал: $\sqrt{\alpha_t}\, z_{t-1}$
2. Добавляем немного шума: $(1 - \alpha_t)\, I$

При $T$ достаточно большом и правильно подобранном расписании $\{\alpha_t\}$, конечное распределение $q(z_T \mid x)$ становится практически неотличимым от $\mathcal{N}(0, I)$, то есть вся информация об $x$ «стирается» шумом.

### Reverse (обратный) процесс — денойзинг

Обратный процесс — это то, что мы реально параметризуем нейросетью:

$$p_\theta(z_{t-1} \mid z_t) = \mathcal{N}\!\left(z_{t-1};\, \mu_\theta(z_t, t),\, \Sigma_\theta(z_t, t)\right)$$

Нейросеть $\mu_\theta$ принимает на вход текущее зашумлённое состояние $z_t$ и номер шага $t$, и предсказывает параметры гауссова перехода на один шаг назад.

Генерация нового объекта тогда сводится к простому алгоритму:

**Алгоритм генерации (DDPM):**
1. Семплировать $z_T \sim \mathcal{N}(0, I)$
2. Для $t = T, T-1, \ldots, 1$: семплировать $z_{t-1} \sim p_\theta(z_{t-1} \mid z_t)$
3. Вернуть $x \approx z_0$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# --- Визуализация forward-процесса на одномерном примере ---
T = 50
betas = np.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)

# "Данные" -- смесь двух гауссиан
n_samples = 2000
data = np.concatenate([
    np.random.randn(n_samples // 2) * 0.5 + 3,
    np.random.randn(n_samples // 2) * 0.5 - 3
])

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
steps_to_show = [0, 5, 10, 20, 49]

for idx, t in enumerate(steps_to_show):
    noise = np.random.randn(len(data))
    z_t = np.sqrt(alpha_bar[t]) * data + np.sqrt(1 - alpha_bar[t]) * noise
    
    axes[0, idx].hist(z_t, bins=80, density=True, alpha=0.7, color=f'C{idx}')
    axes[0, idx].set_title(f'$t = {t}$,  $\\bar{{\\alpha}}_t = {alpha_bar[t]:.3f}$', fontsize=11)
    axes[0, idx].set_xlim(-6, 6)
    axes[0, idx].set_ylim(0, 0.85)

# Нижний ряд: показать signal-to-noise ratio
t_range = np.arange(T)
snr = alpha_bar / (1 - alpha_bar)

axes[1, 0].plot(t_range, alpha_bar, 'C0-', lw=2)
axes[1, 0].set_xlabel('$t$')
axes[1, 0].set_ylabel(r'$\bar{\alpha}_t$')
axes[1, 0].set_title(r'Cumulative $\bar{\alpha}_t$', fontsize=11)

axes[1, 1].plot(t_range, np.sqrt(alpha_bar), 'C1-', lw=2, label=r'signal $\sqrt{\bar\alpha_t}$')
axes[1, 1].plot(t_range, np.sqrt(1 - alpha_bar), 'C3--', lw=2, label=r'noise $\sqrt{1-\bar\alpha_t}$')
axes[1, 1].set_xlabel('$t$')
axes[1, 1].legend(fontsize=9)
axes[1, 1].set_title('Signal vs noise', fontsize=11)

axes[1, 2].semilogy(t_range, snr, 'C2-', lw=2)
axes[1, 2].set_xlabel('$t$')
axes[1, 2].set_ylabel('SNR')
axes[1, 2].set_title('Signal-to-Noise Ratio', fontsize=11)

for i in [3, 4]:
    axes[1, i].axis('off')

plt.suptitle('Forward process: gradual noising', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Forward-процесс на реальных изображениях MNIST

Посмотрим, как forward-процесс работает на реальных цифрах. Используя свернутую форму $q(z_t \mid x) = \mathcal{N}(\sqrt{\bar\alpha_t}\, x,\, (1-\bar\alpha_t)\, I)$, мы можем зашумить изображение до произвольного шага $t$ за одну операцию:

In [ ]:
import torch
import torchvision
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt
import numpy as np

# Load a few MNIST digits
data = MNIST('data/mnist', download=True, train=True)
images = data.data[:8].float() / 255.0 * 2 - 1  # (8, 28, 28), [-1, 1]
images = images.unsqueeze(1)  # (8, 1, 28, 28)

# VP schedule
T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

# Show noising at different timesteps
timesteps = [0, 50, 100, 200, 500, 999]
fig, axes = plt.subplots(8, len(timesteps), figsize=(len(timesteps) * 2, 16))

for col, t in enumerate(timesteps):
    for row in range(8):
        x = images[row]
        eps = torch.randn_like(x)
        z_t = alpha_bar[t].sqrt() * x + (1 - alpha_bar[t]).sqrt() * eps
        img = (z_t.squeeze().numpy() + 1) / 2  # back to [0, 1]
        axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[0, col].set_title(f't = {t}', fontsize=12)

plt.suptitle('Forward process on MNIST: gradual noising', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Заметим характерную иерархию: при малых $t$ ($t < 100$) изменения практически незаметны — цифры чётко читаются. При средних $t$ ($\sim 200{-}500$) контуры ещё угадываются, но детали уже потеряны. При $t \to T$ изображение неотличимо от чистого шума.

На верхнем ряду видно, как двумодальное распределение данных постепенно «растворяется» в гауссовом шуме. На нижнем — динамика ключевых параметров: кумулятивный $\bar\alpha_t$ монотонно убывает, соотношение сигнал/шум (SNR) падает экспоненциально.

## 1.3 Вывод ELBO для цепочки латентных переменных

Поскольку диффузионная модель — это частный случай латентной модели с расширенным латентным пространством $z_{1:T}$, мы можем вывести вариационную нижнюю оценку (ELBO) стандартным образом.

### Вывод

Начнём с лог-правдоподобия и применим неравенство Йенсена:

$$\log p_\theta(x) = \log \int p_\theta(x, z_{1:T}) \, dz_{1:T} = \log \int \frac{p_\theta(x, z_{1:T})}{q(z_{1:T} \mid x)} \, q(z_{1:T} \mid x) \, dz_{1:T}$$

$$\geq \mathbb{E}_{q(z_{1:T} \mid x)} \left[\log \frac{p_\theta(x, z_{1:T})}{q(z_{1:T} \mid x)}\right] =: \mathcal{L}_{\text{ELBO}}(x; \theta)$$

Подставляя марковские разложения $p_\theta(x, z_{1:T}) = p(z_T) \prod_{t=1}^{T} p_\theta(z_{t-1} \mid z_t)$ и $q(z_{1:T} \mid x) = \prod_{t=1}^{T} q(z_t \mid z_{t-1})$ (при $z_0 = x$), после алгебраических преобразований получаем:

$$\mathcal{L}_{\text{ELBO}}(x; \theta) = \underbrace{\mathbb{E}_{q(z_1|x)}\left[\log p_\theta(x \mid z_1)\right]}_{-L_0\text{ (реконструкция)}} - \underbrace{\text{KL}\!\left(q(z_T \mid x) \,\|\, p(z_T)\right)}_{L_T\text{ (prior matching)}} - \underbrace{\sum_{t=2}^{T} \mathbb{E}_{q(z_t|x)}\left[\text{KL}\!\left(q(z_{t-1} \mid z_t, x) \,\|\, p_\theta(z_{t-1} \mid z_t)\right)\right]}_{\sum_{t=2}^{T} L_{t-1}\text{ (denoising matching)}}$$

Разберём три слагаемых:

**$L_0$ (реконструкция)** — аналог реконструкционного слагаемого в VAE. Отвечает за качество восстановления объекта из первого (почти чистого) латента $z_1$. Как правило, реализуется тривиально или отдельным декодером.

**$L_T$ (prior matching)** — KL-дивергенция между $q(z_T \mid x)$ и прайором $p(z_T) = \mathcal{N}(0, I)$. При достаточно большом $T$ и правильном расписании $\bar\alpha_T \approx 0$, так что $q(z_T \mid x) \approx \mathcal{N}(0, I)$ и это слагаемое практически равно нулю. **Не зависит от параметров $\theta$** — не вносит вклад в градиент.

**$L_{t-1}$ (denoising matching)** — основная сумма. Каждое слагаемое требует, чтобы обученный обратный переход $p_\theta(z_{t-1} \mid z_t)$ был близок к «истинному» обратному переходу $q(z_{t-1} \mid z_t, x)$. Именно эта сумма и является основным обучающим функционалом диффузионной модели.

### Сравнение с ELBO для VAE

| | VAE | Диффузионная модель |
|---|---|---|
| Латентные переменные | $z$ (одна) | $z_1, z_2, \ldots, z_T$ (цепочка) |
| Прайор | $p(z) = \mathcal{N}(0, I)$ | $p(z_T) = \mathcal{N}(0, I)$ |
| Энкодер | $q_\phi(z \mid x)$ — **обучаемый** | $q(z_{1:T} \mid x)$ — **фиксированный** |
| Декодер | $p_\theta(x \mid z)$ — один шаг | $\prod_{t=1}^T p_\theta(z_{t-1} \mid z_t)$ — $T$ шагов |
| ELBO | $\mathbb{E}[\log p_\theta(x \mid z)] - \text{KL}(q_\phi \| p)$ | $-L_0 - L_T - \sum L_{t-1}$ |
| Что оптимизируем | $\theta$ (декодер) и $\phi$ (энкодер) | Только $\theta$ (обратный процесс) |

Если положить $T = 1$, то диффузионная модель в точности сводится к VAE с фиксированным энкодером $q(z_1 \mid x) = \mathcal{N}(\sqrt{\alpha_1}\, x,\, (1-\alpha_1)\, I)$.

### VE vs VP процессы

Помимо VP-расписания существует альтернативное семейство — **VE (Variance Exploding)**:

| Свойство | VP (Variance Preserving) | VE (Variance Exploding) |
|---|---|---|
| Свернутая форма | $q(z_t \mid x) = \mathcal{N}(\sqrt{\bar\alpha_t}\, x,\, (1-\bar\alpha_t)\, I)$ | $q(z_t \mid x) = \mathcal{N}(x,\, \sigma_t^2\, I)$ |
| Среднее | Затухает: $\sqrt{\bar\alpha_t} \to 0$ | Постоянное: $x$ |
| Дисперсия | Ограничена: $\leq 1$ | Растёт: $\sigma_t^2 \to \infty$ |
| Конечное распределение | $\mathcal{N}(0, I)$ | $\mathcal{N}(0, \sigma_{\max}^2 I)$ |
| Стабильность | Более стабильное обучение | Может быть нестабильно при больших $\sigma$ |

В VP-процессе сигнал затухает одновременно с ростом шума, так что общая дисперсия остаётся примерно единичной (отсюда имя «variance preserving»). В VE-процессе среднее не меняется, а дисперсия растёт неограниченно.

На практике VP-процессы используются чаще благодаря более стабильной оптимизации. Важное теоретическое наблюдение: оба процесса могут быть преобразованы друг в друга заменой параметров, поэтому модель, обученная с VP, в принципе может использоваться с VE-расписанием.

## 1.4 Гауссов прямой процесс: VP-schedule

### Свернутая форма

Огромное преимущество гауссова ядра: композиция нескольких гауссовых переходов — тоже гауссов переход. Определим кумулятивный параметр:

$$\bar\alpha_t = \prod_{s=1}^{t} \alpha_s$$

Тогда свернутая форма прямого процесса позволяет **зашумить объект сразу до произвольного шага $t$**:

$$q(z_t \mid x) = \mathcal{N}\!\left(z_t;\, \sqrt{\bar\alpha_t}\, x,\, (1 - \bar\alpha_t)\, I\right)$$

Это записывается через репараметризацию:

$$z_t = \sqrt{\bar\alpha_t}\, x + \sqrt{1 - \bar\alpha_t}\, \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

Данное свойство критически важно на практике: при обучении нам не нужно последовательно применять все $t$ шагов — мы сразу семплируем $(x, t, \varepsilon)$ и получаем $z_t$.

### Обратный переход при известном $x$

Для вычисления KL-дивергенций в ELBO нам нужен «истинный» обратный переход $q(z_{t-1} \mid z_t, x)$. По формуле Байеса:

$$q(z_{t-1} \mid z_t, x) = \frac{q(z_t \mid z_{t-1}) \, q(z_{t-1} \mid x)}{q(z_t \mid x)}$$

Поскольку все три распределения гауссовы, их произведение тоже гауссово:

$$q(z_{t-1} \mid z_t, x) = \mathcal{N}\!\left(z_{t-1};\, \tilde\mu_t(x, z_t),\, \tilde\beta_t\, I\right)$$

где:

$$\tilde\mu_t(x, z_t) = \frac{\sqrt{\bar\alpha_{t-1}}\,(1 - \alpha_t)}{1 - \bar\alpha_t}\, x \;+\; \frac{\sqrt{\alpha_t}\,(1 - \bar\alpha_{t-1})}{1 - \bar\alpha_t}\, z_t$$

$$\tilde\beta_t = \frac{(1 - \bar\alpha_{t-1})}{(1 - \bar\alpha_t)}\,(1 - \alpha_t)$$

Заметим, что среднее $\tilde\mu_t$ — линейная функция от $x$ и $z_t$, а дисперсия $\tilde\beta_t$ зависит только от расписания шума и не зависит от данных.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

# --- Реализация VP forward process ---
T = 1000
beta_start, beta_end = 1e-4, 0.02
betas = np.linspace(beta_start, beta_end, T)
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)

# Визуализация зашумления одной "картинки" (случайный вектор)
np.random.seed(42)
x = np.random.randn(28 * 28) * 0.3 + 0.5
x = np.clip(x, 0, 1)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
timesteps = [0, 50, 100, 250, 500, 999]

for i, t in enumerate(timesteps):
    eps = np.random.randn(*x.shape)
    z_t = np.sqrt(alpha_bar[t]) * x + np.sqrt(1 - alpha_bar[t]) * eps
    img = z_t.reshape(28, 28)
    axes[0, i].imshow(img, cmap='gray', vmin=-2, vmax=2)
    axes[0, i].set_title(f't = {t}', fontsize=11)
    axes[0, i].axis('off')

axes[1, 0].plot(betas, 'C0-', lw=1.5)
axes[1, 0].set_xlabel('$t$'); axes[1, 0].set_title(r'$\beta_t$', fontsize=11)

axes[1, 1].plot(alpha_bar, 'C1-', lw=1.5)
axes[1, 1].set_xlabel('$t$'); axes[1, 1].set_title(r'$\bar{\alpha}_t$', fontsize=11)

axes[1, 2].plot(np.sqrt(alpha_bar), label='signal', lw=1.5)
axes[1, 2].plot(np.sqrt(1 - alpha_bar), '--', label='noise', lw=1.5)
axes[1, 2].legend(fontsize=9); axes[1, 2].set_xlabel('$t$'); axes[1, 2].set_title('Signal vs noise', fontsize=11)

beta_tilde = np.zeros(T)
beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * betas[1:]
axes[1, 3].plot(beta_tilde, 'C3-', lw=1.5)
axes[1, 3].set_xlabel('$t$'); axes[1, 3].set_title(r'$\tilde\beta_t$', fontsize=11)

for i in [4, 5]: axes[1, i].axis('off')
plt.suptitle('VP Noise Schedule (T=1000)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

### Cosine schedule

Линейное расписание $\beta_t$ — простейший выбор, но не оптимальный. Nichol & Dhariwal (2021) предложили **cosine schedule**, при котором $\bar\alpha_t$ убывает по косинусоидальному закону:

$$\bar\alpha_t = \frac{f(t)}{f(0)}, \quad f(t) = \cos\left(\frac{t/T + s}{1 + s} \cdot \frac{\pi}{2}\right)^2$$

где $s = 0.008$ — малый сдвиг, предотвращающий слишком малый $\bar\alpha_T$ в окрестности $t=0$.

Основное преимущество: cosine schedule распределяет «информационное уничтожение» более равномерно по шагам. Линейное расписание тратит начальные шаги впустую (шум почти не заметен), а в конце слишком резко переходит к чистому шуму.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = 1000
t = np.arange(T)

# Linear schedule
beta_lin = np.linspace(1e-4, 0.02, T)
alpha_bar_lin = np.cumprod(1 - beta_lin)

# Cosine schedule
s = 0.008
f = np.cos((t / T + s) / (1 + s) * np.pi / 2) ** 2
alpha_bar_cos = f / f[0]
alpha_bar_cos = np.clip(alpha_bar_cos, 1e-5, 1.0)
beta_cos = np.zeros(T)
beta_cos[1:] = 1 - alpha_bar_cos[1:] / alpha_bar_cos[:-1]
beta_cos = np.clip(beta_cos, 0, 0.999)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(t, alpha_bar_lin, 'C0-', lw=2, label='linear')
axes[0].plot(t, alpha_bar_cos, 'C1-', lw=2, label='cosine')
axes[0].set_xlabel('$t$'); axes[0].set_ylabel(r'$\bar{\alpha}_t$')
axes[0].set_title(r'$\bar{\alpha}_t$', fontsize=13); axes[0].legend()

axes[1].plot(t, beta_lin, 'C0-', lw=2, label='linear')
axes[1].plot(t, beta_cos, 'C1-', lw=2, label='cosine')
axes[1].set_xlabel('$t$'); axes[1].set_ylabel(r'$\beta_t$')
axes[1].set_title(r'$\beta_t$', fontsize=13); axes[1].legend()

# SNR comparison
snr_lin = alpha_bar_lin / (1 - alpha_bar_lin + 1e-10)
snr_cos = alpha_bar_cos / (1 - alpha_bar_cos + 1e-10)
axes[2].semilogy(t, snr_lin, 'C0-', lw=2, label='linear')
axes[2].semilogy(t, snr_cos, 'C1-', lw=2, label='cosine')
axes[2].set_xlabel('$t$'); axes[2].set_ylabel('SNR')
axes[2].set_title('Signal-to-Noise Ratio', fontsize=13); axes[2].legend()

plt.suptitle('Linear vs Cosine noise schedule', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

На графике видно, что cosine schedule обеспечивает более плавное убывание SNR. Линейное расписание «тратит» первые ~200 шагов почти без изменений, а затем резко роняет SNR. Cosine schedule распределяет информационное содержание более равномерно, что на практике улучшает качество генерации, особенно на малых разрешениях (как MNIST).

## 1.5 Параметризация и noise-prediction loss

### Три способа параметризации

Промежуточные слагаемые ELBO имеют вид:

$$L_{t-1} \propto \mathbb{E}_{x, z_t}\left[\frac{1}{2\tilde\beta_t} \left\|\tilde\mu_t(x, z_t) - \mu_\theta(z_t, t)\right\|^2\right]$$

Вопрос: **что именно предсказывает нейросеть** $\mu_\theta(z_t, t)$? Поскольку $\tilde\mu_t(x, z_t)$ — линейная функция от $(x, z_t)$, а $z_t = \sqrt{\bar\alpha_t}\, x + \sqrt{1-\bar\alpha_t}\,\varepsilon$, существуют три естественных параметризации:

### $\mu$-параметризация

Сеть непосредственно аппроксимирует среднее обратного перехода: $\mu_\theta(z_t, t) \approx \tilde\mu_t(x, z_t)$

**Недостаток:** целевая величина $\tilde\mu_t$ сильно зависит от $t$ — задача регрессии неоднородна по времени.

### $x_0$-параметризация

Сеть предсказывает $\hat{x}_\theta(z_t, t) \approx x$

**Интерпретация:** сеть пытается восстановить чистый объект из зашумлённого.

### $\varepsilon$-параметризация (стандарт де-факто)

Подставляя $x = \frac{z_t - \sqrt{1-\bar\alpha_t}\,\varepsilon}{\sqrt{\bar\alpha_t}}$ в формулу для $\tilde\mu_t$, получаем:

$$\tilde\mu_t(x, z_t) = \frac{1}{\sqrt{\alpha_t}}\left(z_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon\right)$$

Сеть $\varepsilon_\theta(z_t, t)$ предсказывает **шум**, который был добавлен:

$$L_{t-1} \propto \mathbb{E}_{x, \varepsilon, t}\left[\left\|\varepsilon - \varepsilon_\theta(z_t, t)\right\|^2\right], \quad z_t = \sqrt{\bar\alpha_t}\, x + \sqrt{1-\bar\alpha_t}\,\varepsilon$$

**Ключевое преимущество:** целевая переменная $\varepsilon \sim \mathcal{N}(0, I)$ **всегда одинаково распределена** вне зависимости от $t$. Вся информация о номере шага кодируется через входы сети $(z_t, t)$, а не через распределение таргета. Это делает задачу регрессии более стационарной.

In [ ]:
# === Численная демонстрация эквивалентности параметризаций ===
import torch
import numpy as np

T = 1000
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

# Возьмём произвольный объект и шаг
x = torch.randn(1, 1, 28, 28)
t = 300  # произвольный шаг

eps_true = torch.randn_like(x)
z_t = alpha_bar[t].sqrt() * x + (1 - alpha_bar[t]).sqrt() * eps_true

# "Истинное" среднее обратного перехода
ab_t = alpha_bar[t]; ab_prev = alpha_bar[t-1]; a_t = alphas[t]; b_t = betas[t]

mu_true = (ab_prev.sqrt() * b_t / (1 - ab_t)) * x + (a_t.sqrt() * (1 - ab_prev) / (1 - ab_t)) * z_t

# 1. mu-параметризация: сеть выдаёт mu_true напрямую
mu_from_mu = mu_true

# 2. x0-параметризация: сеть выдаёт x, пересчитываем mu
x0_pred = x  # "идеальное" предсказание
mu_from_x0 = (ab_prev.sqrt() * b_t / (1 - ab_t)) * x0_pred + (a_t.sqrt() * (1 - ab_prev) / (1 - ab_t)) * z_t

# 3. eps-параметризация: сеть выдаёт eps, пересчитываем mu
eps_pred = eps_true  # "идеальное" предсказание
mu_from_eps = (1 / a_t.sqrt()) * (z_t - (b_t / (1 - ab_t).sqrt()) * eps_pred)

print("Все три параметризации дают одно и то же среднее:")
print(f"  mu-param:  mean = {mu_from_mu.mean().item():.6f}, std = {mu_from_mu.std().item():.6f}")
print(f"  x0-param:  mean = {mu_from_x0.mean().item():.6f}, std = {mu_from_x0.std().item():.6f}")
print(f"  eps-param: mean = {mu_from_eps.mean().item():.6f}, std = {mu_from_eps.std().item():.6f}")
print(f"  max |mu - x0|: {(mu_from_mu - mu_from_x0).abs().max().item():.2e}")
print(f"  max |mu - eps|: {(mu_from_mu - mu_from_eps).abs().max().item():.2e}")

Все три параметризации математически эквивалентны: при «идеальном» предсказании они дают одно и то же среднее $\tilde\mu_t(x, z_t)$. Различие — в том, насколько удобна задача регрессии для нейросети. Как мы обсудили, $\varepsilon$-параметризация выигрывает благодаря стационарности целевой переменной.

### Упрощённый loss (Ho et al., 2020)

На практике отбрасывают веса $\frac{1}{2\tilde\beta_t}$ и используют **простую** функцию потерь:

$$L_{\text{simple}} = \mathbb{E}_{x \sim p^*, \,\varepsilon \sim \mathcal{N}(0,I), \,t \sim \text{Uniform}\{1,\ldots,T\}} \left[\left\|\varepsilon - \varepsilon_\theta\!\left(\sqrt{\bar\alpha_t}\, x + \sqrt{1-\bar\alpha_t}\,\varepsilon,\; t\right)\right\|^2\right]$$

Несмотря на то что такой loss не является в точности ELBO (из-за отброшенных весов), эмпирически он приводит к лучшему качеству генерации.

### Алгоритм обучения DDPM

```
Повторять до сходимости:
    1. x ~ p*(x)                          # батч из датасета
    2. t ~ Uniform{1, ..., T}
    3. eps ~ N(0, I)
    4. z_t = sqrt(alpha_bar_t) * x + sqrt(1 - alpha_bar_t) * eps
    5. gradient step on ||eps - eps_theta(z_t, t)||^2
```

### Алгоритм генерации DDPM

```
1. z_T ~ N(0, I)
2. for t = T, T-1, ..., 1:
    a. if t > 1: sigma ~ N(0, I), else sigma = 0
    b. z_{t-1} = (1/sqrt(alpha_t)) * (z_t - (1-alpha_t)/sqrt(1-alpha_bar_t) * eps_theta(z_t, t)) + sqrt(beta_t) * sigma
3. return x = z_0
```

## 1.6 Архитектура: U-Net для денойзинга

Нейросеть $\varepsilon_\theta(z_t, t)$ принимает на вход зашумлённый объект $z_t$ той же размерности, что и данные, и номер шага $t$, и возвращает предсказанный шум той же размерности. Для изображений стандартная архитектура — **U-Net**, адаптированный для задачи денойзинга.

### Почему именно U-Net

1. **Skip connections** — сохраняют высокочастотные детали, которые иначе теряются при прохождении через bottleneck. Для задачи восстановления шума это критически важно.

2. **Encoder-decoder структура** — позволяет работать на разных масштабах разрешения.

3. **Вход и выход одной размерности** — по конструкции U-Net даёт выход той же формы, что и вход.

### Time embedding

Номер шага $t$ кодируется через **синусоидальный позиционный эмбеддинг** (по аналогии с Transformer):

$$\text{PE}(t, 2i) = \sin\!\left(\frac{t}{10000^{2i/d}}\right), \quad \text{PE}(t, 2i+1) = \cos\!\left(\frac{t}{10000^{2i/d}}\right)$$

Этот вектор затем пропускается через MLP и прибавляется к активациям на каждом уровне U-Net.

### Attention в U-Net

В средних и нижних уровнях разрешения добавляются блоки **self-attention**, позволяющие модели учитывать глобальные зависимости.

## 1.7 Реализация DDPM

Перейдём к полной реализации DDPM для генерации изображений MNIST.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST
import torchvision
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# === Data ===
data_train = MNIST('data/mnist', download=True, train=True)
data_test  = MNIST('data/mnist', download=True, train=False)

# Normalize to [-1, 1]
X_train = data_train.data.float().unsqueeze(1) / 255.0 * 2 - 1
X_test  = data_test.data.float().unsqueeze(1)  / 255.0 * 2 - 1

train_loader = DataLoader(TensorDataset(X_train), batch_size=128, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test),  batch_size=128, shuffle=False)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Данные нормализованы в $[-1, 1]$ — это стандартная практика для диффузионных моделей. В отличие от VAE, где мы часто работали с $[0, 1]$ и Bernoulli-декодером, здесь вход и выход модели живут в непрерывном пространстве, и симметричная нормализация обеспечивает лучшую совместимость с гауссовым прайором $\mathcal{N}(0, I)$.

In [ ]:
# === Noise Schedule ===
class NoiseSchedule:
    '''VP (variance-preserving) linear noise schedule.'''
    
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.T = T
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.alpha_bar_prev = F.pad(self.alpha_bar[:-1], (1, 0), value=1.0)
        self.posterior_variance = self.betas * (1.0 - self.alpha_bar_prev) / (1.0 - self.alpha_bar)
    
    def q_sample(self, x0, t, noise=None):
        '''Forward process: sample z_t from q(z_t | x_0).'''
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha_bar = self.alpha_bar[t].sqrt()[:, None, None, None]
        sqrt_one_minus = (1.0 - self.alpha_bar[t]).sqrt()[:, None, None, None]
        return sqrt_alpha_bar * x0 + sqrt_one_minus * noise, noise
    
schedule = NoiseSchedule(T=1000, device=device)
print(f'Schedule: T={schedule.T}, beta=[{schedule.betas[0]:.5f}, {schedule.betas[-1]:.4f}]')

Класс `NoiseSchedule` инкапсулирует все параметры расписания: $\beta_t$, $\alpha_t$, $\bar\alpha_t$, а также дисперсию обратного перехода $\tilde\beta_t$. Метод `q_sample` реализует свернутую форму forward-процесса — именно эта операция будет вызываться на каждом шаге обучения.

In [ ]:
# === Sinusoidal time embedding ===
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


# === Residual block with time conditioning ===
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Sequential(nn.GroupNorm(8, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.conv2 = nn.Sequential(nn.GroupNorm(8, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    
    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.shortcut(x)


# === Simple U-Net for MNIST (28x28) ===
class SimpleUNet(nn.Module):
    '''Simplified U-Net for DDPM on MNIST. Encoder: 28->14->7, Decoder: 7->14->28.'''
    def __init__(self, in_channels=1, base_channels=64, time_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_dim),
            nn.Linear(time_dim, time_dim * 2), nn.SiLU(), nn.Linear(time_dim * 2, time_dim))
        C = base_channels
        # Encoder
        self.enc1 = ResBlock(in_channels, C, time_dim)
        self.down1 = nn.Conv2d(C, C, 3, stride=2, padding=1)
        self.enc2 = ResBlock(C, C * 2, time_dim)
        self.down2 = nn.Conv2d(C * 2, C * 2, 3, stride=2, padding=1)
        # Bottleneck
        self.bottleneck = ResBlock(C * 2, C * 2, time_dim)
        # Decoder
        self.up2 = nn.ConvTranspose2d(C * 2, C * 2, 4, stride=2, padding=1)
        self.dec2 = ResBlock(C * 4, C, time_dim)
        self.up1 = nn.ConvTranspose2d(C, C, 4, stride=2, padding=1)
        self.dec1 = ResBlock(C * 2, C, time_dim)
        # Output
        self.out = nn.Sequential(nn.GroupNorm(8, C), nn.SiLU(), nn.Conv2d(C, in_channels, 1))
    
    def forward(self, x, t):
        '''Predict noise eps_theta(z_t, t). x: (B,C,H,W), t: (B,).'''
        t_emb = self.time_mlp(t)
        h1 = self.enc1(x, t_emb)
        h2 = self.enc2(self.down1(h1), t_emb)
        h = self.bottleneck(self.down2(h2), t_emb)
        h = self.up2(h)
        h = self.dec2(torch.cat([h, h2], 1), t_emb)
        h = self.up1(h)
        h = self.dec1(torch.cat([h, h1], 1), t_emb)
        return self.out(h)

model = SimpleUNet(in_channels=1, base_channels=64, time_dim=128).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

Архитектура нашей U-Net достаточно компактна для MNIST: три уровня разрешения ($28 \to 14 \to 7$), residual blocks с time conditioning через сложение, skip connections через конкатенацию. Для более сложных данных (CIFAR-10, ImageNet) добавляют self-attention блоки, больше каналов и больше уровней.

In [ ]:
# === Training loop ===
def train_ddpm(model, schedule, train_loader, num_epochs=20, lr=2e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for epoch in range(num_epochs):
        model.train()
        epoch_loss, n_batches = 0, 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for (x0,) in pbar:
            x0 = x0.to(device)
            B = x0.shape[0]
            t = torch.randint(0, schedule.T, (B,), device=device)
            noise = torch.randn_like(x0)
            z_t, _ = schedule.q_sample(x0, t, noise)
            noise_pred = model(z_t, t)
            loss = F.mse_loss(noise_pred, noise)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        print(f'  Epoch {epoch+1}: avg loss = {avg_loss:.4f}')
    return losses

# Uncomment to train:
# losses = train_ddpm(model, schedule, train_loader, num_epochs=20, lr=2e-4)

Обратите внимание на простоту цикла обучения: на каждой итерации мы семплируем случайный шаг $t$, зашумляем батч, предсказываем шум и считаем MSE. Никаких adversarial-потерь (как в GAN), никакого KL-дивергенции энкодера (как в VAE) — только один простой $L_2$-loss.

**Совет:** для MNIST достаточно ~15-20 эпох на GPU. Для быстрого теста можно уменьшить `T` до 200 и `base_channels` до 32.

In [ ]:
# === DDPM Sampling ===
@torch.no_grad()
def sample_ddpm(model, schedule, n_samples=64, return_intermediates=False):
    '''DDPM sampling: z_T ~ N(0,I) -> z_{T-1} -> ... -> z_0.'''
    model.eval()
    shape = (n_samples, 1, 28, 28)
    z = torch.randn(shape, device=device)
    intermediates = [z.cpu()] if return_intermediates else None
    for t in tqdm(reversed(range(schedule.T)), total=schedule.T, desc='Sampling'):
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps_pred = model(z, t_batch)
        alpha_t = schedule.alphas[t]
        alpha_bar_t = schedule.alpha_bar[t]
        beta_t = schedule.betas[t]
        mean = (1.0 / alpha_t.sqrt()) * (z - (beta_t / (1 - alpha_bar_t).sqrt()) * eps_pred)
        if t > 0:
            noise = torch.randn_like(z)
            sigma = schedule.posterior_variance[t].sqrt()
            z = mean + sigma * noise
        else:
            z = mean
        if return_intermediates and t % 100 == 0:
            intermediates.append(z.cpu())
    if return_intermediates:
        return z.cpu(), intermediates
    return z.cpu()

# Uncomment after training:
# samples = sample_ddpm(model, schedule, n_samples=64)
# samples = (samples + 1) / 2  # [-1,1] -> [0,1]
# fig = plt.figure(figsize=(10, 10))
# grid = torchvision.utils.make_grid(samples, nrow=8, pad_value=1)
# plt.imshow(grid.permute(1,2,0).numpy(), cmap='gray')
# plt.axis('off'); plt.title('DDPM: generated MNIST'); plt.show()

In [ ]:
# === Визуализация кривой обучения ===
# Раскомментируйте после обучения:

# fig, ax = plt.subplots(figsize=(8, 4))
# ax.plot(losses, 'C0-', lw=2)
# ax.set_xlabel('Epoch', fontsize=12)
# ax.set_ylabel('MSE Loss', fontsize=12)
# ax.set_title('DDPM Training Loss', fontsize=14)
# ax.grid(alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# === Визуализация пошагового расшумления ===
# Раскомментируйте после обучения:

# samples, intermediates = sample_ddpm(model, schedule, n_samples=8, return_intermediates=True)
# 
# n_steps_shown = len(intermediates)
# fig, axes = plt.subplots(8, n_steps_shown, figsize=(2.5 * n_steps_shown, 18))
# step_labels = ['T'] + [f'{1000 - i*100}' for i in range(1, n_steps_shown)]
# 
# for row in range(8):
#     for col in range(n_steps_shown):
#         img = (intermediates[col][row, 0] + 1) / 2
#         axes[row, col].imshow(img.numpy(), cmap='gray', vmin=0, vmax=1)
#         axes[row, col].axis('off')
#     axes[0, col].set_title(f'step {step_labels[col]}', fontsize=10)
# 
# plt.suptitle('Reverse process: from noise to digits', fontsize=14, y=1.01)
# plt.tight_layout()
# plt.show()

### Интерполяция в пространстве шума

Одно из интересных свойств диффузионных моделей: если мы зафиксируем два начальных шума $z_T^{(1)}$ и $z_T^{(2)}$ и будем линейно интерполировать между ними, то reverse-процесс даст плавный переход между соответствующими объектами:

$$z_T^{(\lambda)} = \lambda\, z_T^{(1)} + (1-\lambda)\, z_T^{(2)}, \quad \lambda \in [0, 1]$$

Для DDIM (детерминированный) это работает особенно хорошо, так как генерация полностью определяется начальным шумом.

In [ ]:
# === Интерполяция в пространстве шума (DDIM) ===
# Раскомментируйте после обучения:

# z1 = torch.randn(1, 1, 28, 28, device=device)
# z2 = torch.randn(1, 1, 28, 28, device=device)
# 
# n_interp = 10
# lambdas = np.linspace(0, 1, n_interp)
# z_interp = torch.cat([lam * z1 + (1 - lam) * z2 for lam in lambdas], dim=0)
# 
# samples_interp = sample_ddim_from_z(model, schedule, z_interp, n_steps=50)
# samples_interp = (samples_interp + 1) / 2
# 
# fig, axes = plt.subplots(1, n_interp, figsize=(2 * n_interp, 2.5))
# for i in range(n_interp):
#     axes[i].imshow(samples_interp[i, 0].numpy(), cmap='gray', vmin=0, vmax=1)
#     axes[i].axis('off')
#     axes[i].set_title(f'{lambdas[i]:.1f}', fontsize=10)
# plt.suptitle('Interpolation in noise space (DDIM)', fontsize=13)
# plt.tight_layout()
# plt.show()

### Визуализация обученной модели

После обучения (~15-20 эпох на GPU) модель генерирует вполне узнаваемые рукописные цифры. Качественное отличие от VAE: сгенерированные изображения **значительно более чёткие** и не страдают от характерного «размытия» VAE.

Процесс генерации визуально нагляден: на ранних шагах ($t \approx T$) мы видим чистый шум, затем постепенно проявляется структура — сначала грубые контуры, потом мелкие детали. Это отражает иерархическую природу: **низкие частоты восстанавливаются первыми**, а **высокие — последними**.

# Часть 2. Диффузия на непрерывном времени

## 2.1 От дискретных шагов к SDE

### Мотивация

В дискретной формулировке мы работали с фиксированным числом шагов $T$ и конкретным расписанием $\{\beta_t\}_{t=1}^T$. Однако при $T \to \infty$ и соответствующем масштабировании $\beta_t \to 0$ дискретная марковская цепь сходится к **непрерывному стохастическому процессу**, описываемому стохастическим дифференциальным уравнением (SDE).

### Forward SDE

Пусть время $t \in [0, 1]$, тогда forward-процесс описывается SDE:

$$dx = f(x, t)\, dt + g(t)\, dw$$

где $f(x, t)$ — **дрифт**, $g(t)$ — **диффузионный коэффициент**, $w$ — стандартный винеровский процесс.

Для VP-расписания:

$$f(x, t) = -\frac{1}{2}\beta(t)\, x, \quad g(t) = \sqrt{\beta(t)}$$

**Интуиция:**
- Дрифт $-\frac{1}{2}\beta(t)\, x$ **сжимает** распределение к нулю
- Диффузия $\sqrt{\beta(t)}\, dw$ **расширяет** распределение, добавляя шум

### Связь с дискретным случаем

Дискретная цепь получается как схема Эйлера-Маруямы с шагом $\Delta t = 1/T$:

$$z_{t+1} - z_t = f(z_t, t)\,\Delta t + g(t)\,\sqrt{\Delta t}\,\varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0, I)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

np.random.seed(42)

# === SDE trajectories for VP-process ===
T_cont = 1.0
N_steps = 1000
dt = T_cont / N_steps
t_grid = np.linspace(0, T_cont, N_steps + 1)

def beta_schedule(t, beta_min=0.1, beta_max=20.0):
    return beta_min + t * (beta_max - beta_min)

# Initial points: mixture of two clusters
n_traj = 50
x0 = np.concatenate([
    np.random.randn(n_traj // 2) * 0.3 + 2.0,
    np.random.randn(n_traj // 2) * 0.3 - 2.0,
])

trajectories = np.zeros((n_traj, N_steps + 1))
trajectories[:, 0] = x0

for step in range(N_steps):
    t = step * dt
    bt = beta_schedule(t)
    drift = -0.5 * bt * trajectories[:, step]
    diffusion = np.sqrt(bt)
    noise = np.random.randn(n_traj)
    trajectories[:, step + 1] = trajectories[:, step] + drift * dt + diffusion * np.sqrt(dt) * noise

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i in range(n_traj):
    color = 'C0' if x0[i] > 0 else 'C1'
    axes[0].plot(t_grid, trajectories[i], color=color, alpha=0.15, lw=0.5)
axes[0].set_xlabel('$t$'); axes[0].set_ylabel('$x_t$')
axes[0].set_title('VP SDE trajectories', fontsize=13)
axes[0].axhline(0, color='gray', ls='--', alpha=0.5)

times_to_show = [0, 0.2, 0.5, 0.8, 1.0]
steps_to_show = [int(t / dt) for t in times_to_show]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(times_to_show)))

for t_val, step, color in zip(times_to_show, steps_to_show, colors):
    vals = trajectories[:, step]
    x_range = np.linspace(-5, 5, 200)
    try:
        kde = gaussian_kde(vals, bw_method=0.3)
        axes[1].plot(x_range, kde(x_range), color=color, lw=2, label=f'$t={t_val}$')
    except:
        pass
axes[1].set_xlabel('$x$'); axes[1].set_title('Marginal distributions $p_t(x)$', fontsize=13)
axes[1].legend(fontsize=10)

t_cont = np.linspace(0, 1, 200)
axes[2].plot(t_cont, [beta_schedule(t) for t in t_cont], 'C2-', lw=2)
axes[2].set_xlabel('$t$'); axes[2].set_ylabel(r'$\beta(t)$')
axes[2].set_title(r'Schedule $\beta(t)$', fontsize=13)

plt.tight_layout(); plt.show()

## 2.2 Уравнение Фоккера-Планка и reverse SDE

### Эволюция плотности

Если $x_t$ подчиняется SDE $dx = f(x,t)\,dt + g(t)\,dw$, то его плотность $p_t(x)$ удовлетворяет **уравнению Фоккера-Планка** (Колмогорова вперёд):

$$\frac{\partial p_t(x)}{\partial t} = -\nabla_x \cdot \bigl[f(x,t)\, p_t(x)\bigr] + \frac{g(t)^2}{2}\,\Delta_x\, p_t(x)$$

### Вывод (идея)

Рассмотрим малый шаг $\Delta t$:

$$x_{t+\Delta t} = x_t + f(x_t, t)\,\Delta t + g(t)\,\sqrt{\Delta t}\,\xi, \quad \xi \sim \mathcal{N}(0, I)$$

Разложив переходную плотность по $\Delta t$ до второго порядка (используя $\mathbb{E}[\xi] = 0$, $\mathbb{E}[\xi\xi^T] = I$), получаем уравнение Фоккера-Планка.

### Более детальный вывод (для одномерного случая)

Рассмотрим одномерный случай для простоты. Пусть $X_{t+\Delta t} = X_t + f(X_t)\,\Delta t + g\,\sqrt{\Delta t}\,\xi$, $\xi \sim \mathcal{N}(0,1)$.

Для произвольной гладкой тест-функции $\varphi(x)$:

$$\mathbb{E}[\varphi(X_{t+\Delta t})] = \mathbb{E}\left[\varphi\left(X_t + f(X_t)\Delta t + g\sqrt{\Delta t}\,\xi\right)\right]$$

Разложим $\varphi$ в ряд Тейлора до второго порядка:

$$\varphi(X_t + \delta) \approx \varphi(X_t) + \varphi'(X_t)\,\delta + \frac{1}{2}\varphi''(X_t)\,\delta^2$$

где $\delta = f\,\Delta t + g\sqrt{\Delta t}\,\xi$. Подставляя и беря матожидание (используя $\mathbb{E}[\xi] = 0$, $\mathbb{E}[\xi^2] = 1$, отбрасывая $O(\Delta t^{3/2})$):

$$\mathbb{E}[\varphi(X_{t+\Delta t})] \approx \mathbb{E}[\varphi(X_t)] + \mathbb{E}[\varphi'(X_t) f(X_t)]\,\Delta t + \frac{g^2}{2}\mathbb{E}[\varphi''(X_t)]\,\Delta t$$

Записывая через плотность $p_t$, интегрируя по частям и переходя к пределу $\Delta t \to 0$, получаем:

$$\frac{\partial p_t}{\partial t} = -\frac{\partial}{\partial x}[f(x)\, p_t] + \frac{g^2}{2}\frac{\partial^2 p_t}{\partial x^2}$$

Это и есть уравнение Фоккера-Планка для одномерного случая. Многомерное обобщение получается заменой производных на $\nabla \cdot$ и $\Delta$.

### Reverse-time SDE

Замечательный результат (Anderson, 1982): для forward SDE

$$dx = f(x,t)\,dt + g(t)\,dw$$

существует **обратный по времени** стохастический процесс:

$$dx = \left[f(x,t) - g(t)^2\,\nabla_x \log p_t(x)\right] dt + g(t)\, d\bar{w}$$

где $\bar{w}$ — винеровский процесс, идущий назад во времени.

Ключевой объект — **score function**:

$$s(x, t) := \nabla_x \log p_t(x)$$

Это градиент логарифма плотности, указывающий направление наибольшего роста плотности. Если мы знаем score function для всех $t$, мы можем численно проинтегрировать reverse SDE и получить сэмплы.

**Роль нейросети:** мы обучаем $s_\theta(x, t) \approx \nabla_x \log p_t(x)$.

## 2.3 Score matching и связь с $\varepsilon$-параметризацией

### Denoising score matching

Для гауссова прямого процесса $q(x_t \mid x) = \mathcal{N}(x_t;\, \sqrt{\bar\alpha_t}\, x,\, (1-\bar\alpha_t)\, I)$:

$$\nabla_{x_t} \log q(x_t \mid x) = -\frac{x_t - \sqrt{\bar\alpha_t}\, x}{1 - \bar\alpha_t} = -\frac{\varepsilon}{\sqrt{1 - \bar\alpha_t}}$$

### Связь score и $\varepsilon$-параметризации

Если нейросеть предсказывает шум $\varepsilon_\theta(x_t, t)$, то score восстанавливается как:

$$s_\theta(x_t, t) = -\frac{\varepsilon_\theta(x_t, t)}{\sqrt{1 - \bar\alpha_t}}$$

То есть **$\varepsilon$-параметризация и score matching — это один и тот же объект** с точностью до масштабирующего множителя! Минимизация noise-prediction loss эквивалентна denoising score matching.

Эта связь объединяет два мира:
- **Вариационный подход** (DDPM): работаем с ELBO, предсказываем шум
- **Score-based подход** (Song & Ermon): работаем со score function, решаем reverse SDE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def gmm_pdf(X, Y, means, stds, weights):
    p = np.zeros_like(X)
    for (mx, my), s, w in zip(means, stds, weights):
        p += w * np.exp(-0.5 * ((X - mx)**2 + (Y - my)**2) / s**2) / (2 * np.pi * s**2)
    return p

def gmm_score(X, Y, means, stds, weights):
    p = gmm_pdf(X, Y, means, stds, weights)
    sx, sy = np.zeros_like(X), np.zeros_like(Y)
    for (mx, my), s, w in zip(means, stds, weights):
        comp = w * np.exp(-0.5 * ((X - mx)**2 + (Y - my)**2) / s**2) / (2 * np.pi * s**2)
        sx += comp * (-(X - mx) / s**2)
        sy += comp * (-(Y - my) / s**2)
    return sx / (p + 1e-10), sy / (p + 1e-10)

means = [(-2, -2), (2, 2), (-2, 2)]
stds = [0.6, 0.6, 0.6]
weights = [0.4, 0.35, 0.25]

grid = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(grid, grid)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
noise_levels = [0.0, 0.5, 1.5]
titles = ['t ~ 0 (clean data)', 't ~ 0.3 (medium noise)', 't ~ 0.8 (heavy noise)']

for ax, sigma_noise, title in zip(axes, noise_levels, titles):
    noisy_stds = [np.sqrt(s**2 + sigma_noise**2) for s in stds]
    p = gmm_pdf(X, Y, means, noisy_stds, weights)
    sx, sy = gmm_score(X, Y, means, noisy_stds, weights)
    ax.contourf(X, Y, p, levels=30, cmap='Blues', alpha=0.5)
    step = 6
    Xq, Yq = X[::step, ::step], Y[::step, ::step]
    Sxq, Syq = sx[::step, ::step], sy[::step, ::step]
    norm = np.sqrt(Sxq**2 + Syq**2) + 1e-10
    ax.quiver(Xq, Yq, Sxq/norm, Syq/norm, norm, cmap='Reds', alpha=0.7, scale=25)
    ax.set_title(title, fontsize=13)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect('equal')

plt.suptitle(r'Score function $\nabla_x \log p_t(x)$: arrows point toward data', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

На левом графике score function при малом шуме указывает к ближайшей моде. По мере увеличения шума моды «размазываются», score поле становится более гладким.

## 2.4 Probability Flow ODE и DDIM

### Детерминированная альтернатива

Для каждой SDE существует **обыкновенное дифференциальное уравнение** (ODE) с теми же маргиналами:

$$\frac{dx}{dt} = f(x,t) - \frac{1}{2}\, g(t)^2\, \nabla_x \log p_t(x)$$

Это **Probability Flow ODE** — детерминированное, без стохастичности. Можно использовать эффективные численные схемы и «перескакивать» через шаги.

### DDIM (Denoising Diffusion Implicit Models)

DDIM (Song et al., 2020) — дискретизация Probability Flow ODE. Формула обновления:

$$z_{\tau_{i-1}} = \sqrt{\bar\alpha_{\tau_{i-1}}}\,\underbrace{\left(\frac{z_{\tau_i} - \sqrt{1-\bar\alpha_{\tau_i}}\,\varepsilon_\theta(z_{\tau_i}, \tau_i)}{\sqrt{\bar\alpha_{\tau_i}}}\right)}_{\text{predicted }x_0} + \sqrt{1 - \bar\alpha_{\tau_{i-1}}}\,\varepsilon_\theta(z_{\tau_i}, \tau_i)$$

Ключевые свойства:
- **Детерминированная** генерация (одна $z_T$ всегда даёт одну $z_0$)
- Можно значительно **сократить** число шагов: 50, 20, даже 10 вместо 1000

In [ ]:
# === DDIM Sampler ===
@torch.no_grad()
def sample_ddim(model, schedule, n_samples=64, n_steps=50):
    '''DDIM sampling with arbitrary number of steps.'''
    model.eval()
    step_indices = np.linspace(0, schedule.T - 1, n_steps, dtype=int)
    step_indices = sorted(set(step_indices), reverse=True)
    z = torch.randn(n_samples, 1, 28, 28, device=device)
    for i in tqdm(range(len(step_indices)), desc=f'DDIM ({n_steps} steps)'):
        t = step_indices[i]
        t_prev = step_indices[i + 1] if i + 1 < len(step_indices) else 0
        t_batch = torch.full((n_samples,), t, device=device, dtype=torch.long)
        eps_pred = model(z, t_batch)
        alpha_bar_t = schedule.alpha_bar[t]
        alpha_bar_prev = schedule.alpha_bar[t_prev] if t_prev > 0 else torch.tensor(1.0)
        x0_pred = (z - (1 - alpha_bar_t).sqrt() * eps_pred) / alpha_bar_t.sqrt()
        z = alpha_bar_prev.sqrt() * x0_pred + (1 - alpha_bar_prev).sqrt() * eps_pred
    return z.cpu()

# Usage after training:
# samples_50  = sample_ddim(model, schedule, n_samples=64, n_steps=50)
# samples_10  = sample_ddim(model, schedule, n_samples=64, n_steps=10)

In [ ]:
# === Сравнение DDIM при разном числе шагов ===
# Раскомментируйте после обучения:

# fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
# step_counts = [1000, 100, 50, 10]
# titles = ['DDPM (1000 steps)', 'DDIM (100 steps)', 'DDIM (50 steps)', 'DDIM (10 steps)']
# 
# # Фиксируем одинаковый начальный шум для сравнения
# torch.manual_seed(42)
# z_T = torch.randn(16, 1, 28, 28, device=device)
# 
# for ax, n_steps, title in zip(axes, step_counts, titles):
#     if n_steps == 1000:
#         # DDPM full
#         samples = sample_ddpm_from_z(model, schedule, z_T.clone())
#     else:
#         samples = sample_ddim_from_z(model, schedule, z_T.clone(), n_steps=n_steps)
#     samples = (samples + 1) / 2
#     grid = torchvision.utils.make_grid(samples, nrow=4, pad_value=1)
#     ax.imshow(grid.permute(1,2,0).numpy(), cmap='gray')
#     ax.set_title(title, fontsize=12)
#     ax.axis('off')
# 
# plt.suptitle('DDPM vs DDIM: speed-quality trade-off', fontsize=14)
# plt.tight_layout()
# plt.show()

### Trade-off: число шагов vs качество

| Число шагов | Время (отн.) | Качество |
|---|---|---|
| 1000 (DDPM) | 1x | Наилучшее |
| 100 (DDIM) | ~10x быстрее | Почти без потерь |
| 50 (DDIM) | ~20x быстрее | Незначительная потеря |
| 10 (DDIM) | ~100x быстрее | Заметная потеря |

# Часть 3. Систематическое сравнение: VAE vs GAN vs Diffusion

| Свойство | VAE | GAN | Диффузия |
|---|---|---|---|
| **Качество сэмплов** | Низкое (размытость) | Высокое | Высокое |
| **Покрытие мод** | Хорошее | Плохое (mode collapse) | Хорошее |
| **Скорость генерации** | Быстрая (1 шаг) | Быстрая (1 шаг) | Медленная ($T$ шагов) |
| **Доступ к плотности** | Есть (ELBO) | Нет | Есть (ELBO) |
| **Стабильность обучения** | Стабильное | Нестабильное | Стабильное |
| **Контролируемость** | Интерполяция в $\mathcal{Z}$ | Слабая | Guidance, условная генерация |

### Когда что использовать

**VAE** — быстрый стабильный кодек. Отлично как компрессор латентных представлений (Stable Diffusion). Не для генерации высокого качества.

**GAN** — быстрая генерация для узких доменов. Требует настройки обучения.

**Диффузия** — приоритет на **качество и разнообразие**. Стандарт для text-to-image, аудио, видео.

In [ ]:
# === Визуальное сравнение VAE / GAN / Diffusion ===
# Раскомментируйте, подставив обученные модели из предыдущих лекций:

# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# 
# # VAE samples (from lecture 2)
# with torch.no_grad():
#     vae_samples = vae.generate_samples(16).cpu().view(-1, 1, 28, 28)
# grid_vae = torchvision.utils.make_grid(vae_samples, nrow=4, pad_value=1)
# axes[0].imshow(grid_vae.permute(1,2,0).numpy(), cmap='gray')
# axes[0].set_title('VAE\n(размытые, но разнообразные)', fontsize=12)
# axes[0].axis('off')
# 
# # GAN samples (from lecture 3)
# with torch.no_grad():
#     z = torch.randn(16, latent_dim, device=device)
#     gan_samples = G(z).cpu()
# grid_gan = torchvision.utils.make_grid(gan_samples, nrow=4, pad_value=1, normalize=True)
# axes[1].imshow(grid_gan.permute(1,2,0).numpy(), cmap='gray')
# axes[1].set_title('GAN\n(чёткие, но mode collapse)', fontsize=12)
# axes[1].axis('off')
# 
# # Diffusion samples
# diff_samples = sample_ddpm(model, schedule, n_samples=16)
# diff_samples = (diff_samples + 1) / 2
# grid_diff = torchvision.utils.make_grid(diff_samples, nrow=4, pad_value=1)
# axes[2].imshow(grid_diff.permute(1,2,0).numpy(), cmap='gray')
# axes[2].set_title('Diffusion\n(чёткие И разнообразные)', fontsize=12)
# axes[2].axis('off')
# 
# plt.suptitle('Сравнение генеративных моделей на MNIST', fontsize=15)
# plt.tight_layout()
# plt.show()

# Часть 4. Диффузия в индустрии

## 4.1 Latent Diffusion и Stable Diffusion

### Проблема: диффузия в пиксельном пространстве

Для изображения $512 \times 512 \times 3$ U-Net работает с тензорами размерности $\sim\!800\,000$ — на каждом из $T$ шагов. Колоссальные вычислительные затраты.

### Идея Latent Diffusion (Rombach et al., 2022)

Проведём диффузию **в латентном пространстве VAE**:

1. **VAE-энкодер** $E$: сжимает $x \in \mathbb{R}^{H \times W \times 3}$ в латент $z = E(x) \in \mathbb{R}^{h \times w \times c}$, где $h = H/8$, $w = W/8$, $c = 4$. Сжатие в $\sim\!48$ раз.
2. **Диффузия в $\mathcal{Z}$**: forward/reverse процесс работает с компактными латентами
3. **VAE-декодер** $D$: $\hat{x} = D(z_0)$ из расшумлённого латента

### Conditioning: текст $\to$ изображение

Текст $c$ кодируется текстовым энкодером (CLIP) в последовательность эмбеддингов $\tau_\theta(c) \in \mathbb{R}^{L \times d_{\text{text}}}$.

В каждом attention-блоке U-Net:
- **Query** — из текущих активаций U-Net (визуальные фичи)
- **Key, Value** — из текстовых эмбеддингов

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### Classifier-Free Guidance

При инференсе:

$$\tilde\varepsilon_\theta(z_t, t, c) = \varepsilon_\theta(z_t, t, \varnothing) + w \cdot \left[\varepsilon_\theta(z_t, t, c) - \varepsilon_\theta(z_t, t, \varnothing)\right]$$

где $w > 1$ — **guidance scale** (типично 7-15). Чем больше $w$, тем сильнее изображение «следует» тексту.

In [ ]:
# === Pipeline diagram (Stable Diffusion) ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(16, 5))

blocks = [
    (0.02, 0.3, 0.12, 0.4, 'Text\n"a cat\nwearing\na hat"', '#FFD700'),
    (0.16, 0.3, 0.12, 0.4, 'CLIP\nText\nEncoder', '#87CEEB'),
    (0.35, 0.1, 0.12, 0.8, 'U-Net\n(latent\nspace)\n\ncross-attn\nwith text\n\nT steps', '#98FB98'),
    (0.55, 0.3, 0.12, 0.4, 'Latent\n$z_0$', '#DDA0DD'),
    (0.72, 0.3, 0.12, 0.4, 'VAE\nDecoder', '#FFA07A'),
    (0.88, 0.3, 0.1, 0.4, 'Image\n512x512', '#FF6347'),
]

for x, y, w, h, text, color in blocks:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.01",
                                     facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9, fontweight='bold')

arrow_style = dict(arrowstyle='->', lw=2, color='black')
for x1, x2, y in [(0.14, 0.16, 0.5), (0.28, 0.35, 0.5), (0.47, 0.55, 0.5),
                   (0.67, 0.72, 0.5), (0.84, 0.88, 0.5)]:
    ax.annotate('', xy=(x2, y), xytext=(x1, y), arrowprops=arrow_style)

ax.text(0.41, 0.95, '$z_T \\sim \\mathcal{N}(0, I)$', ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
ax.annotate('', xy=(0.41, 0.9), xytext=(0.41, 0.92), arrowprops=arrow_style)

ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.05, 1.05)
ax.set_aspect('auto'); ax.axis('off')
ax.set_title('Stable Diffusion Pipeline (Latent Diffusion Model)', fontsize=14, pad=20)
plt.tight_layout(); plt.show()

## 4.2 CLIP: связь текста и изображений

### Что такое CLIP

**CLIP** (Contrastive Language-Image Pre-training, Radford et al., 2021) — модель, обученная на ~400 млн пар (текст, изображение) из интернета.

Архитектура:
- **Image Encoder** $f_I(x)$: ViT или ResNet $\to \mathbb{R}^d$
- **Text Encoder** $f_T(c)$: Transformer $\to \mathbb{R}^d$

### Contrastive Loss

Для батча из $N$ пар $(x_i, c_i)$:

$$s_{ij} = \frac{f_I(x_i)^T f_T(c_j)}{\|f_I(x_i)\| \cdot \|f_T(c_j)\|}$$

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[\log\frac{\exp(s_{ii}/\tau)}{\sum_j \exp(s_{ij}/\tau)} + \log\frac{\exp(s_{ii}/\tau)}{\sum_j \exp(s_{ji}/\tau)}\right]$$

В Stable Diffusion CLIP text encoder кодирует промпты, подаваемые через cross-attention в U-Net.

In [ ]:
# === Иллюстрация контрастивного обучения CLIP ===
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
N = 6  # batch size

# Simulate similarity matrix
similarity = np.random.randn(N, N) * 0.3
# Make diagonal (matching pairs) high
for i in range(N):
    similarity[i, i] = 2.0 + np.random.rand() * 0.5

# Apply softmax temperature
tau = 0.07
exp_sim = np.exp(similarity / tau)
probs = exp_sim / exp_sim.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Raw similarities
im0 = axes[0].imshow(similarity, cmap='RdBu_r', vmin=-1, vmax=3)
axes[0].set_title('Cosine similarity $s_{ij}$', fontsize=12)
axes[0].set_xlabel('Text'); axes[0].set_ylabel('Image')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Softmax probabilities
im1 = axes[1].imshow(probs, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('After softmax (row-wise)', fontsize=12)
axes[1].set_xlabel('Text'); axes[1].set_ylabel('Image')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Target: identity matrix
target = np.eye(N)
im2 = axes[2].imshow(target, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title('Target: matching pairs', fontsize=12)
axes[2].set_xlabel('Text'); axes[2].set_ylabel('Image')
plt.colorbar(im2, ax=axes[2], shrink=0.8)

for ax in axes:
    ax.set_xticks(range(N)); ax.set_yticks(range(N))

plt.suptitle('CLIP: contrastive learning on (image, text) pairs', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

CLIP обучается так, чтобы правильные пары (изображение $i$, текст $i$) имели максимальное сходство (диагональ матрицы), а неправильные пары — минимальное. После обучения текстовый и визуальный энкодеры «понимают» одну семантику: эмбеддинг текста «кот» оказывается близок к эмбеддингам фотографий котов.

Именно это свойство делает CLIP идеальным мостом между текстовыми промптами и диффузионной генерацией.

## 4.3 Эволюция DALL-E

### DALL-E 1 (Ramesh et al., 2021)

1. **Дискретный VAE (dVAE):** изображение кодируется в $32 \times 32$ дискретных токенов
2. **Авторегрессия:** GPT-подобная модель (12 млрд параметров) генерирует токены
3. **CLIP** для reranking сгенерированных вариантов

### DALL-E 2 (Ramesh et al., 2022)

Переход к **диффузионному** подходу:
1. **CLIP:** текст и изображение в общее эмбеддинг-пространство
2. **Prior:** диффузия переводит текстовый CLIP-эмбеддинг в изображенческий
3. **Decoder (unCLIP):** вторая диффузия генерирует изображение по CLIP-эмбеддингу

### DALL-E 3 (Betker et al., 2023)

1. Обучен специальный **captioner** для подробных описаний изображений
2. Модель обучена на парах (подробное описание, изображение) — критически улучшает следование промпту
3. T5 для текстового кодирования + диффузия в латентном пространстве

Эволюция DALL-E показывает путь от авторегрессии к диффузии, и значимость качества данных.

## 4.4 Современное состояние и тренды

### Ускорение инференса

**Distillation** — обучить «ученика» (меньше шагов) имитировать «учителя». Progressive distillation: итеративно сокращают число шагов вдвое.

**Consistency Models** (Song et al., 2023) — модель напрямую отображает любую точку траектории ODE в $x_0$. Генерация за 1-2 шага.

**Rectified Flow / Flow Matching** (Lipman et al., 2022) — ODE с прямолинейными траекториями, мало шагов без дополнительных ухищрений.

### Генерация видео

**Sora** (OpenAI, 2024), **Runway Gen-3**, **Stable Video Diffusion** — пространственно-временной U-Net или DiT (Diffusion Transformer), temporal attention для согласованности кадров.

### За пределами изображений

- **Аудио:** AudioLDM, MusicGen
- **3D:** DreamFusion, Magic3D — генерация 3D через score distillation
- **Молекулы:** Equivariant diffusion для дизайна белков
- **Текст:** Diffusion-LM, MDLM — дискретная диффузия (активная область исследований)

### Полный пайплайн генерации Stable Diffusion

Соединив все компоненты, получаем следующий алгоритм text-to-image генерации:

```
Вход: текстовый промпт c, guidance scale w, число шагов S

1. Кодировать текст: emb = CLIP_text_encoder(c)
2. Семплировать начальный латент: z_T ~ N(0, I)   (размер h x w x 4)
3. Для каждого шага s = S, S-1, ..., 1:
    a. Предсказать шум дважды:
       eps_cond   = UNet(z_s, s, emb)        # условное предсказание
       eps_uncond = UNet(z_s, s, empty_emb)   # безусловное предсказание
    b. Classifier-free guidance:
       eps_guided = eps_uncond + w * (eps_cond - eps_uncond)
    c. DDIM/DDPM шаг: z_{s-1} = denoise_step(z_s, eps_guided)
4. Декодировать латент: image = VAE_decoder(z_0)
```

Обратите внимание: на каждом шаге денойзинга U-Net вызывается **дважды** (условно и безусловно). Это основная причина, почему classifier-free guidance удваивает время инференса.

# Заключение

На этой лекции мы прошли путь от VAE к диффузионным моделям, показав, что диффузия — это **естественное обобщение** латентного вероятностного моделирования:

1. **Дискретное время:** диффузия = VAE с цепочкой латентных переменных. ELBO раскладывается на сумму KL-дивергенций, каждая из которых сводится к задаче денойзинга.

2. **Параметризация:** $\varepsilon$-параметризация делает задачу регрессии стационарной по времени.

3. **Непрерывное время:** при $T \to \infty$ получаем SDE/ODE формулировку. Score function напрямую связана с $\varepsilon$-параметризацией.

4. **Практика:** DDIM для ускорения, Latent Diffusion + CLIP для text-to-image, эволюция от DALL-E 1 к DALL-E 3.

Диффузионные модели совмещают **высокое качество** с **хорошим покрытием мод** и **теоретической обоснованностью**, что сделало их стандартом в генеративном AI.

# Ресурсы

### Основные статьи

- **[DDPM]** Ho, Jain, Abbeel. *Denoising Diffusion Probabilistic Models* (NeurIPS 2020) — https://arxiv.org/abs/2006.11239
- **[Score SDE]** Song et al. *Score-Based Generative Modeling through SDE* (ICLR 2021) — https://arxiv.org/abs/2011.13456
- **[DDIM]** Song, Meng, Ermon. *Denoising Diffusion Implicit Models* (ICLR 2021) — https://arxiv.org/abs/2010.02502
- **[Latent Diffusion]** Rombach et al. *High-Resolution Image Synthesis with Latent Diffusion Models* (CVPR 2022) — https://arxiv.org/abs/2112.10752
- **[CLIP]** Radford et al. *Learning Transferable Visual Models From Natural Language Supervision* (ICML 2021) — https://arxiv.org/abs/2103.00020
- **[DALL-E 2]** Ramesh et al. *Hierarchical Text-Conditional Image Generation with CLIP Latents* — https://arxiv.org/abs/2204.06125
- **[Classifier-Free Guidance]** Ho, Salimans. *Classifier-Free Diffusion Guidance* — https://arxiv.org/abs/2207.12598

### Туториалы и обзоры

- Lilian Weng. *What are Diffusion Models?* — https://lilianweng.github.io/posts/2021-07-11-diffusion-models/
- Yang Song. *Generative Modeling by Estimating Gradients of the Data Distribution* — https://yang-song.net/blog/2021/score/
- Calvin Luo. *Understanding Diffusion Models: A Unified Perspective* — https://arxiv.org/abs/2208.11970